# Face Liveness Detection — Google Colab Training

**COSC-4117EL-01 Assignment 3**

This notebook trains and evaluates the face liveness detection models on a GPU.
All persistent outputs (checkpoints, metrics, plots) are saved to Google Drive.

**Prerequisites:**
- Upload the Kaggle dataset zip to `My Drive/liveness/kaggle_dataset.zip`
- Upload the `Assignment_3/` source code to `My Drive/liveness/code/`
  (or clone from GitHub with `!git clone ...`)

**Workflow:**
1. Setup & install dependencies
2. Mount Drive, unzip dataset
3. Extract frames, preprocess (face crop), generate splits
4. Train all models (ResNet-18/34/50, YOLO11n/s)
5. Evaluate and compare
6. Export best model to ONNX

## Cell 1 — Environment Setup

In [ ]:
# Verify GPU
!nvidia-smi

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q \
    torchvision ultralytics opencv-python-headless \
    mediapipe onnx onnxruntime \
    scikit-learn matplotlib seaborn pandas PyYAML tqdm albumentations

## Cell 2 — Mount Google Drive & Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
from pathlib import Path

# === CONFIGURE THESE PATHS ===
DRIVE_ROOT    = Path('/content/drive/MyDrive/liveness')
CODE_DIR      = DRIVE_ROOT / 'code'          # Assignment_3/ code
DATASET_ZIP   = DRIVE_ROOT / 'kaggle_dataset.zip'
DATA_ROOT     = DRIVE_ROOT / 'data'
RESULTS_ROOT  = DRIVE_ROOT / 'experiments'

# Add source code to Python path
sys.path.insert(0, str(CODE_DIR))

print(f"Code     : {CODE_DIR}")
print(f"Data     : {DATA_ROOT}")
print(f"Results  : {RESULTS_ROOT}")

# Verify code is accessible
assert (CODE_DIR / 'src' / 'config.py').exists(), \
    f"Source code not found at {CODE_DIR}. Upload Assignment_3/ to Google Drive."

## Cell 3 — Dataset Preparation

In [ ]:
# Unzip Kaggle dataset (skip if already done)
RAW_DIR = DATA_ROOT / 'raw' / 'kaggle_videos'

if not RAW_DIR.exists() or not any(RAW_DIR.rglob('*.mp4')):
    print("Unzipping dataset...")
    import zipfile
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        zf.extractall(str(RAW_DIR))
    print("Done.")
else:
    print("Dataset already extracted.")

# Show structure
for cls in ('real', 'fake'):
    vids = list((RAW_DIR / cls).rglob('*.mp4')) if (RAW_DIR / cls).exists() else []
    print(f"  {cls}: {len(vids)} videos")

In [ ]:
# Extract frames from videos
FRAMES_DIR = DATA_ROOT / 'frames'

if not FRAMES_DIR.exists() or not any(FRAMES_DIR.rglob('*.jpg')):
    !python {CODE_DIR}/scripts/extract_frames.py \
        --input-root  {RAW_DIR} \
        --output-root {FRAMES_DIR} \
        --source kaggle \
        --fps-sample 2 \
        --max-frames 500
else:
    print("Frames already extracted.")

for cls in ('real', 'fake'):
    imgs = list((FRAMES_DIR / cls).glob('*.jpg'))
    print(f"  {cls} frames: {len(imgs)}")

In [ ]:
# Face detect, crop, resize → processed/
PROCESSED_DIR = DATA_ROOT / 'processed'

if not PROCESSED_DIR.exists() or not any(PROCESSED_DIR.rglob('*.jpg')):
    !python {CODE_DIR}/scripts/preprocess_dataset.py \
        --input-root  {FRAMES_DIR} \
        --output-root {PROCESSED_DIR} \
        --backend mediapipe \
        --image-size 224 \
        --source kaggle
else:
    print("Dataset already preprocessed.")

for cls in ('real', 'fake'):
    imgs = list((PROCESSED_DIR / cls).glob('*.jpg'))
    print(f"  {cls} processed: {len(imgs)}")

In [ ]:
# Generate stratified train/val/test splits
SPLITS_DIR = DATA_ROOT / 'splits'

if not (SPLITS_DIR / 'train.csv').exists():
    !python {CODE_DIR}/scripts/generate_splits.py \
        --processed-root {PROCESSED_DIR} \
        --output-dir     {SPLITS_DIR} \
        --train 0.70 --val 0.15 --test 0.15 \
        --seed 42
else:
    print("Splits already generated.")
    import pandas as pd
    for split in ('train', 'val', 'test'):
        df = pd.read_csv(SPLITS_DIR / f'{split}.csv')
        print(f"  {split}: {len(df)} samples")

## Cell 4 — Training

In [ ]:
# === Experiment 1a: ResNet-18 Baseline ===
# Expected: ~15-25 min on T4 GPU
!python {CODE_DIR}/train.py \
    --config {CODE_DIR}/experiments/configs/resnet18_baseline.yaml

In [ ]:
# === Experiment 1b: ResNet-34 ===
!python {CODE_DIR}/train.py \
    --config {CODE_DIR}/experiments/configs/resnet34.yaml

In [ ]:
# === Experiment 1c: ResNet-50 ===
!python {CODE_DIR}/train.py \
    --config {CODE_DIR}/experiments/configs/resnet50.yaml

In [ ]:
# === Experiment 2: Dataset size impact ===
for fraction, epochs in [(0.10, 20), (0.25, 25), (0.50, 28), (1.00, 30)]:
    run_name = f"resnet18_subset_{fraction:.2f}"
    print(f"\n{'='*60}")
    print(f"Training subset={fraction} → {run_name}")
    print('='*60)
    !python {CODE_DIR}/train.py \
        --config   {CODE_DIR}/experiments/configs/resnet18_subset_experiments.yaml \
        --subset   {fraction} \
        --epochs   {epochs} \
        --run-name {run_name}

In [ ]:
# === Experiment 5: YOLO11 comparison ===
!python {CODE_DIR}/train.py \
    --config {CODE_DIR}/experiments/configs/yolo11n_cls.yaml

!python {CODE_DIR}/train.py \
    --config {CODE_DIR}/experiments/configs/yolo11s_cls.yaml

## Cell 5 — Evaluation & Comparison

In [ ]:
# Cross-experiment comparison table
!python {CODE_DIR}/evaluate.py --compare {RESULTS_ROOT}/results

# Display in notebook
import pandas as pd
from IPython.display import display

df = pd.read_csv(RESULTS_ROOT / 'results' / 'comparison_table.csv')
display(df.style.highlight_max(
    subset=['accuracy', 'f1', 'fps'],
    color='#d4f0d4'
).format(precision=4))

In [ ]:
# Plot comparison bar charts
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(RESULTS_ROOT / 'results' / 'comparison_table.csv')

# Filter to main model comparison (Experiment 1 + 5)
main_models = df[df['run_name'].str.contains('full_run|yolo')].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ['accuracy', 'f1', 'fps']):
    if metric in main_models.columns:
        sub = main_models.dropna(subset=[metric])
        ax.bar(sub['model'], sub[metric], color='steelblue')
        ax.set_title(metric.upper())
        ax.set_ylim(0, 1.1 if metric != 'fps' else None)
        ax.tick_params(axis='x', rotation=30)
        ax.grid(axis='y', alpha=0.3)

plt.suptitle('Model Comparison — Experiments 1 & 5', fontsize=13)
plt.tight_layout()
plt.savefig(str(RESULTS_ROOT / 'results' / 'model_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Display confusion matrices for all main runs
from IPython.display import Image, display
from pathlib import Path

for cm_path in sorted((RESULTS_ROOT / 'results').rglob('confusion_matrix.png')):
    print(cm_path.parent.name)
    display(Image(str(cm_path), width=400))

In [ ]:
# Display training curves for ResNet models
for curve_path in sorted((RESULTS_ROOT / 'results').rglob('training_curves.png')):
    print(curve_path.parent.name)
    display(Image(str(curve_path), width=700))

## Cell 6 — ONNX Export

In [ ]:
# Export best ResNet-18 to ONNX
BEST_CKPT  = RESULTS_ROOT / 'results' / 'resnet18_full_run1' / 'weights' / 'best.pt'
ONNX_OUT   = DRIVE_ROOT / 'export' / 'model.onnx'
ONNX_OUT.parent.mkdir(parents=True, exist_ok=True)

!python {CODE_DIR}/export_onnx.py \
    --checkpoint {BEST_CKPT} \
    --model resnet18 \
    --output {ONNX_OUT}

print(f"ONNX model size: {ONNX_OUT.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Download ONNX model and preprocessing meta to local machine
from google.colab import files

files.download(str(ONNX_OUT))
files.download(str(ONNX_OUT.parent / 'preprocessing_meta.json'))
print("Download initiated. Place files in Assignment_3/export/")

## Cell 7 — Summary

In [ ]:
import pandas as pd

df = pd.read_csv(RESULTS_ROOT / 'results' / 'comparison_table.csv')

print('=' * 70)
print('EXPERIMENT RESULTS SUMMARY')
print('=' * 70)

# Experiment 1: Model comparison
print('\nExperiment 1 — ResNet Architecture Comparison:')
exp1 = df[df['model'].isin(['resnet18','resnet34','resnet50'])]
print(exp1[['model','accuracy','f1','fps','mean_latency_ms']].to_string(index=False))

# Experiment 2: Dataset size
print('\nExperiment 2 — Dataset Size Impact (ResNet-18):')
exp2 = df[df['run_name'].str.contains('subset')].sort_values('subset_fraction')
if not exp2.empty:
    print(exp2[['subset_fraction','accuracy','f1']].to_string(index=False))

# Experiment 5: YOLO comparison
print('\nExperiment 5 — ResNet vs YOLO Comparison:')
exp5 = df[df['model'].isin(['resnet18','yolo11n-cls','yolo11s-cls'])]
print(exp5[['model','accuracy','f1','fps']].to_string(index=False))

print('\n' + '=' * 70)
best = df.loc[df['f1'].idxmax()]
print(f"Best model: {best['model']} — F1={best['f1']:.4f}, Acc={best['accuracy']:.4f}")